In [1]:
!wget https://phasepro.elte.hu/download_full.json -O downloads/phasepro.json

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [55]:
import json
phasepro_dict = json.load(open("downloads/phasepro.json"))
import pandas as pd
df = pd.DataFrame.from_dict(phasepro_dict, orient='index')

In [56]:
import goatools
from goatools.obo_parser import GODag
go_dag = GODag("downloads/go-basic.obo", optional_attrs=['def', 'synonym'])

downloads/go-basic.obo: fmt(1.2) rel(2025-06-01) 43,448 Terms; optional_attrs(def synonym)


In [57]:
organelles = []
org_name_col = []
for organelle_str in df['organelles']:
    # Split the string by semi-colons and strip whitespace
    org_names = [org.strip() for org in organelle_str.split(';') if org.strip()]
    org_name_col.append(org_names)
    organelles.extend(org_names)
organelles = list(set(organelles))  # Remove duplicates
df['org_names'] = org_name_col

In [58]:
org_go_map = {'synaptosome, neuron projection': 'GO:0043005',
 'membrane cluster' : 'GO:0032991',
 'a matrix holding together clusters of synaptic vesicles (SVs)': 'GO:0099523',
 'intracellular DNA/protein granule': 'GO:0043232',
 'Ftsz-rich droplets at specific chromosomal DNA sites': 'GO:0016604', 
 'bacterial RNP-bodies (BR-bodies)': 'GO:0036464',
 'Sup35 condensate': 'GO:0043232',
 'selective hydrogel-like meshwork formed by FG-nucleoporins in nuclear pore central channel': 'GO:0044613',
 'POLII clusters': 'GO:0016591',
 'cGAS foci': 'GO:0043232',
 'Sec body': 'GO:0043232',
 'axonal TIAR-2 granules': 'GO:0043232',
 'condensed compartments of microtubule bundling': 'GO:0005856',
 'robustly E2-activated enhancers (MegaTrans) enhancers': 'GO:0034206',
 'MORC3-NBs': 'GO:0016604',
 'Sam68 nuclear bodies (SNBs)': 'GO:0016604',
 'viroplasm': 'GO:0039716',
 'collagen-containing extracellular matrix': 'GO:0031012',
 'Std1 bodies': 'GO:0043232',
 'liquid-like DYRK3 speckles': 'GO:0043232',
 'basal Numb-Pon crescent in dividing neuroblasts': 'GO:0043232',
 'SPOP/DAXX body': 'GO:0043232',
 'paraspeckle': 'GO:0042382',
 'cytoplasmic protein granule': 'GO:0043232',
 'p62 body': 'GO:0016234',
 'nuclear bodies that occur at super enhancers in mESCs': 'GO:0016604',
 'Negri body': 'GO:0039714',
 'small RNA amplification complex': 'GO:1990633',
 'PcG chromatin condensates': 'GO:0031519',
 'nuclear protein granule': 'GO:0043232',
 'miRISC': 'GO:0016442'}
org_go_map = {k: [v] for k, v in org_go_map.items()}

In [59]:
for organelle in organelles:
    # print(organelle)
    if(not organelle in org_go_map):
        org_go_map[organelle] = []
    for term in go_dag:
        if(go_dag[term].is_obsolete):
            continue
        if organelle.lower() == go_dag[term].name.lower():
            org_go_map[organelle].append(term)
            # print(f"Adding GO term: {term} - {go_dag[term].name}")
        elif any(organelle.lower() == synonym.text.lower() for synonym in go_dag[term].synonym):
            org_go_map[organelle].append(term)
            # print(f"Adding GO term: {term} - {go_dag[term].name} using synonym")
        # elif organelle.lower() in go_dag[term].defn.lower():
        #     print(f"Adding GO term: {term} - {go_dag[term].name} using desc")

In [60]:
go_term_col = []
for org in df['org_names']:
    terms = []
    for o in org:
        if o in org_go_map:
            terms.extend(org_go_map[o])
    terms = list(set(terms))  # Remove duplicates
    go_term_col.append(terms)
df['go_terms'] = go_term_col

In [ ]:
df['boundaries'] = df['boundaries'].apply(lambda x: '[' + x.replace(';', ',') + ']')
def parse_boundaries(boundaries):
    boundaries = boundaries.replace('[', '').replace(']', '')
    chunks = [b.strip() for b in boundaries.split(',')]
    ret = []
    for chunk in chunks:
        if '-' in chunk:
            start, end = chunk.split('-')
            ret.append((int(start.strip()), int(end.strip())))
        else:
            ret.append(int(chunk))
    return ret
df['boundaries'] = df['boundaries'].apply(parse_boundaries)

In [63]:
llps_dataset = df[['accession', 'boundaries', 'go_terms', 'sequence']]
llps_dataset.columns = ['UniprotID', 'AnnotatedIndices', 'GOTerm', 'Sequence']
llps_dataset.to_csv('datasets/llps_dataset.csv', index=False, sep='\t')

In [1]:
import pandas as pd
dataset_df = pd.read_csv('datasets/llps_dataset.csv', sep='\t').dropna()
dataset_df = dataset_df[[len(t) < 850 for t in dataset_df['Sequence']]]
dataset_df.to_csv('datasets/llps_dataset.csv', index=False, sep='\t')